# TA-03 — Data-Driven Monte Carlo Simulation
**Topik:** Estimasi Risiko Klaim Asuransi (digit terakhir NIM = 4)
**Nama:** Ari Mauludin | **NIM:** 301240044 | **Kelas:** 5C
**Mata Praktikum:** Pemodelan dan Simulasi

**Pertanyaan bisnis:** Jika ada 100 pasien baru, berapa probabilitas total klaim melebihi anggaran perusahaan? (fitur `charges`)

**Dataset:** Medical Cost Personal Datasets (Kaggle, file `insurance.csv`, 1.338 baris).
Letakkan `insurance.csv` di folder yang sama dengan notebook ini, atau upload saat diminta di Google Colab.

## Langkah 0 — Import library & seed (NIM)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Seed memakai NIM agar hasil konsisten / reproducible
np.random.seed(301240044)

## Langkah 1 — Eksplorasi Data Kaggle
### 1.1 Load data

In [ ]:
CSV = "insurance.csv"
if not os.path.exists(CSV):
    try:
        from google.colab import files          # khusus Google Colab
        print("Upload file insurance.csv ...")
        uploaded = files.upload()
        CSV = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError("Letakkan insurance.csv di folder yang sama dengan notebook ini.")

df = pd.read_csv(CSV)
print("Ukuran data:", df.shape)
df.head()

### 1.2 Data cleaning
Hapus nilai null, duplikat, dan **outlier ekstrem** (batas IQR × 3). Memakai batas ekstrem (bukan 1,5×IQR) karena klaim besar pada perokok adalah data riil yang justru penting untuk analisis risiko; yang dibuang hanya nilai yang benar-benar ekstrem.

In [ ]:
print("Jumlah null per kolom:\n", df.isnull().sum(), sep="")

n_awal = len(df)
df = df.dropna().drop_duplicates()
print(f"\nBaris dibuang (null/duplikat): {n_awal - len(df)}")

q1, q3 = df["charges"].quantile([0.25, 0.75])
iqr = q3 - q1
batas_bawah, batas_atas = q1 - 3 * iqr, q3 + 3 * iqr
df_clean = df[(df["charges"] >= batas_bawah) & (df["charges"] <= batas_atas)].copy()

print(f"Batas outlier ekstrem charges: {batas_bawah:,.2f} s/d {batas_atas:,.2f}")
print(f"Outlier ekstrem dibuang      : {len(df) - len(df_clean)}")
print(f"Data bersih                  : {len(df_clean)} baris")

### 1.3 Parameter Extraction (variabel acak = `charges`)

In [ ]:
charges = df_clean["charges"]
mu = charges.mean()
sigma = charges.std(ddof=1)

print(f"Mean (μ)            : {mu:,.2f}")
print(f"Std Dev (σ)         : {sigma:,.2f}")
print(f"Skewness            : {stats.skew(charges):.3f}")
print(f"Min / Median / Max  : {charges.min():,.2f} / {charges.median():,.2f} / {charges.max():,.2f}")

### 1.4 Histogram data asli

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist(charges, bins=40, density=True, color="steelblue", alpha=0.75, label="Data asli (charges)")
xs = np.linspace(0, charges.max(), 400)
plt.plot(xs, stats.norm.pdf(xs, mu, sigma), "r-", lw=2, label=f"Normal(μ={mu:,.0f}, σ={sigma:,.0f})")
plt.title("Distribusi Biaya Medis (charges)")
plt.xlabel("charges"); plt.ylabel("Densitas"); plt.legend()
plt.show()

**Interpretasi:** histogram condong ke kanan (right-skewed), sehingga distribusi Normal hanya pendekatan kasar untuk *satu* pasien. Namun untuk **total 100 pasien**, Teorema Limit Pusat membuat distribusi jumlahnya mendekati Normal. Pada langkah berikutnya, model Normal (sesuai modul) dibandingkan dengan Lognormal dan Bootstrap sebagai uji sensitivitas.

## Langkah 2 — Membangun Simulasi Monte Carlo

In [ ]:
N_SCENARIOS = 10_000     # jumlah skenario (minimal 5.000)
N_PATIENTS  = 100        # pasien baru per skenario

# Asumsi anggaran: ekspektasi total klaim + buffer 10%
BUFFER = 0.10
BUDGET = (1 + BUFFER) * N_PATIENTS * mu
print(f"Ekspektasi total klaim 100 pasien : {N_PATIENTS * mu:,.2f}")
print(f"Anggaran perusahaan (buffer {BUFFER:.0%})   : {BUDGET:,.2f}")

# Model utama: Normal(μ, σ) sesuai ketentuan modul.
# Tidak di-clip: pemotongan di 0 membuat rata-rata simulasi bias ke atas (lihat 3.3).
klaim = np.random.normal(mu, sigma, size=(N_SCENARIOS, N_PATIENTS))
total_klaim = klaim.sum(axis=1)       # total klaim tiap skenario
print("Bentuk array klaim:", klaim.shape)

## Langkah 3 — Analisis Risiko & Kesimpulan
### 3.1 Probabilitas
$P(Event)=\dfrac{\text{Jumlah skenario total klaim} > \text{anggaran}}{\text{Total skenario}}$

In [ ]:
jumlah_melebihi = np.sum(total_klaim > BUDGET)
prob = jumlah_melebihi / N_SCENARIOS
se = np.sqrt(prob * (1 - prob) / N_SCENARIOS)

print(f"Skenario total klaim > anggaran : {jumlah_melebihi} dari {N_SCENARIOS}")
print(f"P(total klaim > anggaran)       : {prob:.4f}  ({prob:.2%})")
print(f"Interval 95% (galat Monte Carlo): {prob - 1.96*se:.4f} - {prob + 1.96*se:.4f}")
print(f"Rata-rata total klaim simulasi  : {total_klaim.mean():,.2f}")
print(f"Std Dev total klaim simulasi    : {total_klaim.std(ddof=1):,.2f}")
print(f"VaR 95% (persentil ke-95)       : {np.percentile(total_klaim, 95):,.2f}")

### 3.2 Visualisasi hasil (histogram + garis threshold)

In [ ]:
plt.figure(figsize=(9, 5))
n, bins, patches = plt.hist(total_klaim, bins=60, color="steelblue", alpha=0.8)
for b, p in zip(bins[:-1], patches):
    if b >= BUDGET:
        p.set_facecolor("crimson")           # area risiko
plt.axvline(BUDGET, color="black", ls="--", lw=2, label=f"Anggaran = {BUDGET:,.0f}")
plt.axvline(total_klaim.mean(), color="green", ls=":", lw=2, label=f"Rata-rata = {total_klaim.mean():,.0f}")
plt.title(f"Distribusi Total Klaim 100 Pasien ({N_SCENARIOS:,} skenario)\nP(total > anggaran) = {prob:.2%}")
plt.xlabel("Total klaim"); plt.ylabel("Jumlah skenario"); plt.legend()
plt.show()

### 3.3 Uji sensitivitas: Normal vs Lognormal vs Bootstrap
**Catatan metodologi:** karena σ hampir sebesar μ, distribusi Normal menghasilkan sebagian klaim negatif. Memotongnya di 0 (`clip`) membuat rata-rata simulasi *lebih besar* daripada rata-rata data asli, sehingga probabilitas risiko overestimasi. Karena itu model utama tidak di-clip, dan versi clip hanya ditampilkan sebagai pembanding. Tabel di bawah membandingkan model utama dengan tiga alternatif agar terlihat seberapa sensitif hasilnya.

In [ ]:
# Lognormal: fit pada log(charges)
log_c = np.log(charges)
total_logn = np.random.lognormal(log_c.mean(), log_c.std(ddof=1), size=(N_SCENARIOS, N_PATIENTS)).sum(axis=1)

# Bootstrap: sampling ulang langsung dari data historis
total_boot = np.random.choice(charges.values, size=(N_SCENARIOS, N_PATIENTS), replace=True).sum(axis=1)

# Normal + clip 0 (klaim negatif dipotong, tetapi rata-rata menjadi bias ke atas)
total_clip = np.clip(np.random.normal(mu, sigma, size=(N_SCENARIOS, N_PATIENTS)), 0, None).sum(axis=1)

hasil = pd.DataFrame({
    "Model": ["Normal (utama)", "Normal + clip 0", "Lognormal", "Bootstrap"],
    "Rata-rata total": [total_klaim.mean(), total_clip.mean(), total_logn.mean(), total_boot.mean()],
    "P(total > anggaran)": [np.mean(total_klaim > BUDGET), np.mean(total_clip > BUDGET), np.mean(total_logn > BUDGET), np.mean(total_boot > BUDGET)],
})
hasil["P(total > anggaran)"] = hasil["P(total > anggaran)"].map("{:.2%}".format)
hasil["Rata-rata total"] = hasil["Rata-rata total"].map("{:,.0f}".format)
hasil

### 3.4 Kesimpulan otomatis dari hasil simulasi

In [ ]:
print("KESIMPULAN")
print("-" * 60)
print(f"Dari {N_SCENARIOS:,} skenario 100 pasien baru, total klaim melebihi anggaran")
print(f"{BUDGET:,.0f} (ekspektasi + buffer {BUFFER:.0%}) sebanyak {jumlah_melebihi} kali,")
print(f"sehingga probabilitas risiko ≈ {prob:.2%}.")
print(f"Dengan keyakinan 95%, total klaim tidak akan melebihi {np.percentile(total_klaim, 95):,.0f}.")
print("Anggaran perlu dinaikkan jika perusahaan menginginkan probabilitas risiko lebih kecil.")